# 01: AWS & Boto3 Fundamentals for ML Engineers

## Module 3, Week 10: Cloud Infrastructure

---

### Learning Objectives
By the end of this notebook, you will:
1. **Understand core AWS services** for ML workflows (S3, IAM, Lambda)
2. **Use boto3** to interact with AWS programmatically
3. **Build S3 data pipelines** — upload, download, list, and organize ML data
4. **Apply IAM best practices** — least-privilege policies for ML roles
5. **Write Lambda handlers** for lightweight ML triggers
6. **Compare storage formats** — CSV vs Parquet for ML datasets

### Resources
- [AWS Free Tier](https://aws.amazon.com/free/) — 5 GB S3 storage, 1M Lambda requests/month
- [Boto3 Documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)
- [AWS ML University (YouTube)](https://www.youtube.com/c/AmazonWebServices)
- [Made With ML — MLOps](https://madewithml.com/)

### How This Notebook Works
All code runs **locally** using a `LocalS3Simulator` class that mirrors the boto3 S3 API.
This means you can learn S3 patterns without an AWS account. Each section also shows the
equivalent real-AWS code in comments or markdown blocks.

Set `USE_REAL_AWS = True` below if you have AWS credentials configured.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import os
import json
import time
import shutil
import hashlib
import tempfile
from pathlib import Path
from datetime import datetime
from io import BytesIO, StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Toggle this to True if you have AWS credentials configured
USE_REAL_AWS = False

if USE_REAL_AWS:
    import boto3
    print(f"boto3 version: {boto3.__version__}")
    sts = boto3.client('sts')
    try:
        identity = sts.get_caller_identity()
        print(f"AWS Account: {identity['Account']}")
        print(f"User ARN:    {identity['Arn']}")
    except Exception as e:
        print(f"AWS not configured: {e}")
        print("Falling back to local simulator.")
        USE_REAL_AWS = False
else:
    print("Running in LOCAL SIMULATOR mode (no AWS credentials needed).")
    print("Set USE_REAL_AWS = True to use real AWS services.")

---
## Section 1: AWS Overview for ML Engineers

AWS is the most widely used cloud platform for ML. As an ML engineer, you don't need to master
every service — but you need **deep familiarity** with these core services:

| Service | What It Does | ML Use Case | Free Tier |
|---------|-------------|-------------|----------|
| **S3** | Object storage | Store datasets, model artifacts, logs | 5 GB / 12 months |
| **IAM** | Identity & access | Control who/what can access resources | Always free |
| **Lambda** | Serverless functions | Trigger pipelines, lightweight inference | 1M requests/month |
| **SageMaker** | ML platform | Training, HPO, deployment | 250 hrs t2.medium/2 months |
| **ECR** | Container registry | Store Docker images for deployment | 500 MB / 12 months |
| **CloudWatch** | Monitoring/logging | Track pipeline health, model metrics | 10 metrics free |

### The Mental Model: Everything is an API

Every AWS service is an HTTP API. The AWS Console (web UI) is just a frontend for those APIs.
**Boto3** is the Python SDK that lets you call these same APIs from code.

```
Your Python Code  →  boto3  →  HTTP API  →  AWS Service
                                              (S3, Lambda, ...)
```

### When to Use AWS for ML

| Scenario | Local | AWS |
|----------|-------|-----|
| Prototyping with < 1GB data | ✅ | Overkill |
| Training on > 10GB data | Slow | ✅ |
| Serving model to users | Not scalable | ✅ |
| One-time analysis | ✅ | Overkill |
| Team collaboration on data | Git/NFS | ✅ S3 |
| GPU training | Expensive hardware | ✅ Pay-per-use |

---
## Section 2: IAM — Identity and Access Management

IAM is the **security backbone** of AWS. Every request to AWS must be authenticated (who are you?)
and authorized (what are you allowed to do?).

### Key Concepts

| Concept | What It Is | Example |
|---------|-----------|--------|
| **User** | A person or service account | Your developer account |
| **Role** | A set of permissions that can be assumed | `SageMakerExecutionRole` |
| **Policy** | JSON document defining permissions | "Allow read from S3 bucket X" |
| **Group** | Collection of users sharing policies | `data-scientists` group |

### The Principle of Least Privilege

> Grant only the permissions needed to perform a task — nothing more.

This is the #1 security rule in AWS. For ML engineers, this means:
- Your training jobs should only access the specific S3 buckets they need
- Lambda functions should have minimal permissions
- Never use root credentials or `AdministratorAccess` for ML workloads

### Anatomy of an IAM Policy

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowS3ReadWrite",
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:PutObject",
                "s3:ListBucket"
            ],
            "Resource": [
                "arn:aws:s3:::my-ml-bucket",
                "arn:aws:s3:::my-ml-bucket/*"
            ]
        }
    ]
}
```

**Read it as:** "Allow the holder of this policy to Get, Put, and List objects in `my-ml-bucket`."

In [ ]:
# Let's explore IAM policy structures programmatically

def create_s3_policy(bucket_name: str, actions: list[str], read_only: bool = False) -> dict:
    """Generate an IAM policy document for S3 access."""
    if read_only:
        actions = ["s3:GetObject", "s3:ListBucket"]
    
    return {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "S3Access",
                "Effect": "Allow",
                "Action": actions,
                "Resource": [
                    f"arn:aws:s3:::{bucket_name}",
                    f"arn:aws:s3:::{bucket_name}/*"
                ]
            }
        ]
    }

# Policy for an ML training job (read raw data, write model artifacts)
training_policy = create_s3_policy(
    bucket_name="my-ml-data",
    actions=["s3:GetObject", "s3:PutObject", "s3:ListBucket"]
)
print("=== Training Job Policy ===")
print(json.dumps(training_policy, indent=2))

# Read-only policy for a dashboard service
dashboard_policy = create_s3_policy(
    bucket_name="my-ml-data",
    actions=[],
    read_only=True
)
print("\n=== Dashboard Read-Only Policy ===")
print(json.dumps(dashboard_policy, indent=2))

In [ ]:
# Common IAM roles for ML workloads

ml_roles = {
    "SageMaker Execution Role": {
        "purpose": "Assumed by SageMaker training jobs and endpoints",
        "needs": ["S3 read/write", "ECR pull", "CloudWatch logs"],
        "anti-pattern": "Don't give it full S3 access — scope to specific buckets"
    },
    "Lambda Execution Role": {
        "purpose": "Assumed by Lambda functions",
        "needs": ["S3 read (trigger bucket)", "S3 write (output bucket)", "CloudWatch logs"],
        "anti-pattern": "Don't give it SageMaker permissions unless it launches training jobs"
    },
    "CI/CD Pipeline Role": {
        "purpose": "Used by GitHub Actions or CodePipeline",
        "needs": ["ECR push", "SageMaker create/update endpoint", "S3 read/write"],
        "anti-pattern": "Don't give it IAM admin — just the permissions for deploy"
    }
}

for role_name, details in ml_roles.items():
    print(f"\n{'='*60}")
    print(f"Role: {role_name}")
    print(f"Purpose: {details['purpose']}")
    print(f"Needs: {', '.join(details['needs'])}")
    print(f"⚠️  Anti-pattern: {details['anti-pattern']}")

---
## Section 3: S3 Core Concepts

**Amazon S3 (Simple Storage Service)** is the backbone of ML data management in AWS.
Think of it as a giant, infinitely scalable filesystem in the cloud.

### Key Concepts

| Concept | Analogy | Example |
|---------|---------|--------|
| **Bucket** | Top-level folder | `my-ml-project-data` |
| **Key** | File path within bucket | `raw/2024-01/customers.csv` |
| **Object** | The actual file (key + data + metadata) | The CSV file itself |
| **Prefix** | "Folder" in S3 (just a naming convention) | `raw/`, `processed/`, `models/` |

### ML Data Organization Pattern

```
s3://my-ml-bucket/
├── raw/                    # Original, untouched data
│   ├── 2024-01/
│   │   ├── customers.csv
│   │   └── transactions.csv
│   └── 2024-02/
│       └── customers.csv
├── processed/              # Cleaned, validated data
│   └── customers_clean.parquet
├── features/               # Feature-engineered datasets
│   └── training_features.parquet
├── models/                 # Trained model artifacts
│   ├── v1/
│   │   ├── model.pkl
│   │   └── metadata.json
│   └── v2/
│       ├── model.pkl
│       └── metadata.json
└── logs/                   # Training logs, metrics
    └── training_run_001.json
```

### Important S3 Rules
1. **Bucket names are globally unique** — no two AWS accounts can have the same bucket name
2. **Objects are immutable** — you can't edit in place, only replace
3. **S3 is eventually consistent** for overwrite PUTs and DELETEs (usually milliseconds)
4. **No actual folders** — prefixes with `/` just look like folders in the UI

In [ ]:
class LocalS3Simulator:
    """Simulates AWS S3 using the local filesystem.
    
    Mirrors the boto3 S3 client API so you can learn S3 patterns
    without needing an AWS account. Switch to real boto3 by setting
    USE_REAL_AWS = True at the top of the notebook.
    
    Supported operations:
        - create_bucket / list_buckets
        - put_object / get_object / delete_object
        - list_objects_v2 (with prefix filtering)
        - upload_file / download_file
        - head_object (metadata)
    """
    
    def __init__(self, root_dir: str = None):
        self.root = Path(root_dir or tempfile.mkdtemp(prefix="local-s3-"))
        self.root.mkdir(parents=True, exist_ok=True)
        self._metadata = {}  # bucket/key -> metadata dict
        print(f"LocalS3Simulator initialized at: {self.root}")
    
    def create_bucket(self, Bucket: str, **kwargs) -> dict:
        """Create a new bucket (local directory)."""
        bucket_path = self.root / Bucket
        bucket_path.mkdir(exist_ok=True)
        return {"Location": f"/{Bucket}"}
    
    def list_buckets(self) -> dict:
        """List all buckets."""
        buckets = []
        for p in sorted(self.root.iterdir()):
            if p.is_dir():
                buckets.append({
                    "Name": p.name,
                    "CreationDate": datetime.fromtimestamp(p.stat().st_ctime)
                })
        return {"Buckets": buckets}
    
    def put_object(self, Bucket: str, Key: str, Body: bytes, **kwargs) -> dict:
        """Upload an object to a bucket."""
        if isinstance(Body, str):
            Body = Body.encode('utf-8')
        
        obj_path = self.root / Bucket / Key
        obj_path.parent.mkdir(parents=True, exist_ok=True)
        obj_path.write_bytes(Body)
        
        # Store metadata
        meta_key = f"{Bucket}/{Key}"
        self._metadata[meta_key] = {
            "ContentLength": len(Body),
            "LastModified": datetime.now(),
            "ETag": hashlib.md5(Body).hexdigest(),
            **kwargs.get("Metadata", {})
        }
        return {"ETag": self._metadata[meta_key]["ETag"]}
    
    def get_object(self, Bucket: str, Key: str) -> dict:
        """Download an object from a bucket."""
        obj_path = self.root / Bucket / Key
        if not obj_path.exists():
            raise FileNotFoundError(f"NoSuchKey: {Key} in bucket {Bucket}")
        
        body = obj_path.read_bytes()
        return {
            "Body": BytesIO(body),
            "ContentLength": len(body),
            "LastModified": datetime.fromtimestamp(obj_path.stat().st_mtime),
            "ETag": hashlib.md5(body).hexdigest()
        }
    
    def delete_object(self, Bucket: str, Key: str) -> dict:
        """Delete an object."""
        obj_path = self.root / Bucket / Key
        if obj_path.exists():
            obj_path.unlink()
        return {}
    
    def head_object(self, Bucket: str, Key: str) -> dict:
        """Get object metadata without downloading."""
        obj_path = self.root / Bucket / Key
        if not obj_path.exists():
            raise FileNotFoundError(f"NoSuchKey: {Key}")
        
        meta_key = f"{Bucket}/{Key}"
        stat = obj_path.stat()
        return self._metadata.get(meta_key, {
            "ContentLength": stat.st_size,
            "LastModified": datetime.fromtimestamp(stat.st_mtime)
        })
    
    def list_objects_v2(self, Bucket: str, Prefix: str = "", MaxKeys: int = 1000) -> dict:
        """List objects in a bucket with optional prefix filter."""
        bucket_path = self.root / Bucket
        if not bucket_path.exists():
            return {"Contents": [], "KeyCount": 0}
        
        objects = []
        for p in sorted(bucket_path.rglob("*")):
            if p.is_file():
                key = str(p.relative_to(bucket_path))
                if key.startswith(Prefix):
                    objects.append({
                        "Key": key,
                        "Size": p.stat().st_size,
                        "LastModified": datetime.fromtimestamp(p.stat().st_mtime)
                    })
        
        objects = objects[:MaxKeys]
        return {
            "Contents": objects,
            "KeyCount": len(objects),
            "IsTruncated": len(objects) == MaxKeys
        }
    
    def upload_file(self, Filename: str, Bucket: str, Key: str) -> None:
        """Upload a local file to a bucket."""
        with open(Filename, 'rb') as f:
            self.put_object(Bucket=Bucket, Key=Key, Body=f.read())
    
    def download_file(self, Bucket: str, Key: str, Filename: str) -> None:
        """Download an object to a local file."""
        response = self.get_object(Bucket=Bucket, Key=Key)
        Path(Filename).parent.mkdir(parents=True, exist_ok=True)
        with open(Filename, 'wb') as f:
            f.write(response['Body'].read())
    
    def get_bucket_size(self, Bucket: str) -> dict:
        """Calculate total size of all objects in a bucket."""
        result = self.list_objects_v2(Bucket=Bucket)
        total_size = sum(obj['Size'] for obj in result.get('Contents', []))
        return {
            "total_objects": result['KeyCount'],
            "total_bytes": total_size,
            "total_mb": round(total_size / (1024 * 1024), 2)
        }
    
    def cleanup(self):
        """Remove all local S3 data."""
        if self.root.exists():
            shutil.rmtree(self.root)
            print(f"Cleaned up: {self.root}")


# Initialize our S3 client
if USE_REAL_AWS:
    s3 = boto3.client('s3')
    print("Using REAL AWS S3 client")
else:
    s3 = LocalS3Simulator()
    print("Using LocalS3Simulator")

---
## Section 4: S3 with Boto3 — CRUD Operations

Let's learn the core S3 operations. The API is the same whether you're using
the `LocalS3Simulator` or real boto3 — that's the point!

### The boto3 S3 API surface you need:

| Operation | Method | When to Use |
|-----------|--------|------------|
| Create bucket | `create_bucket()` | One-time setup |
| Upload data | `put_object()` / `upload_file()` | Storing data and models |
| Download data | `get_object()` / `download_file()` | Loading data for training |
| List objects | `list_objects_v2()` | Discovering available data |
| Check existence | `head_object()` | Validate before processing |
| Delete | `delete_object()` | Cleanup old artifacts |

In [ ]:
# --- Create a bucket ---
BUCKET_NAME = "ml-demo-bucket"
s3.create_bucket(Bucket=BUCKET_NAME)
print(f"Created bucket: {BUCKET_NAME}")

# List buckets
response = s3.list_buckets()
print(f"\nAll buckets:")
for bucket in response['Buckets']:
    print(f"  - {bucket['Name']} (created: {bucket['CreationDate']})")

In [ ]:
# --- Upload objects ---

# Upload a JSON config
config = {
    "model_type": "gradient_boosting",
    "target": "churned",
    "features": ["tenure_months", "monthly_charges", "contract"],
    "created_at": datetime.now().isoformat()
}
s3.put_object(
    Bucket=BUCKET_NAME,
    Key="config/model_config.json",
    Body=json.dumps(config, indent=2).encode('utf-8')
)
print("Uploaded: config/model_config.json")

# Upload a DataFrame as CSV
df_sample = pd.DataFrame({
    'customer_id': range(1, 101),
    'tenure_months': np.random.randint(1, 72, 100),
    'monthly_charges': np.round(np.random.uniform(20, 100, 100), 2),
    'churned': np.random.binomial(1, 0.25, 100)
})

csv_buffer = df_sample.to_csv(index=False).encode('utf-8')
s3.put_object(
    Bucket=BUCKET_NAME,
    Key="raw/2024-01/customers.csv",
    Body=csv_buffer
)
print(f"Uploaded: raw/2024-01/customers.csv ({len(csv_buffer):,} bytes)")

In [ ]:
# --- Download and read objects ---

# Download the config
response = s3.get_object(Bucket=BUCKET_NAME, Key="config/model_config.json")
config_downloaded = json.loads(response['Body'].read().decode('utf-8'))
print("Downloaded config:")
print(json.dumps(config_downloaded, indent=2))

# Download the CSV as a DataFrame
response = s3.get_object(Bucket=BUCKET_NAME, Key="raw/2024-01/customers.csv")
df_downloaded = pd.read_csv(BytesIO(response['Body'].read()))
print(f"\nDownloaded DataFrame: {df_downloaded.shape}")
print(df_downloaded.head())

In [ ]:
# --- List objects with prefix filtering ---

# Upload a few more files to make listing interesting
for month in ['2024-02', '2024-03']:
    df_month = pd.DataFrame({
        'customer_id': range(1, 51),
        'tenure_months': np.random.randint(1, 72, 50),
        'monthly_charges': np.round(np.random.uniform(20, 100, 50), 2),
        'churned': np.random.binomial(1, 0.25, 50)
    })
    csv_bytes = df_month.to_csv(index=False).encode('utf-8')
    s3.put_object(Bucket=BUCKET_NAME, Key=f"raw/{month}/customers.csv", Body=csv_bytes)

# List ALL objects
response = s3.list_objects_v2(Bucket=BUCKET_NAME)
print("All objects in bucket:")
for obj in response.get('Contents', []):
    print(f"  {obj['Key']:40s}  {obj['Size']:>8,} bytes")

# List only raw/ prefix
print("\nObjects under raw/ prefix:")
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="raw/")
for obj in response.get('Contents', []):
    print(f"  {obj['Key']:40s}  {obj['Size']:>8,} bytes")

In [ ]:
# --- Check object metadata (head_object) ---

meta = s3.head_object(Bucket=BUCKET_NAME, Key="raw/2024-01/customers.csv")
print("Object metadata:")
print(f"  Size: {meta['ContentLength']:,} bytes")
print(f"  Last Modified: {meta['LastModified']}")

# --- Delete an object ---
s3.delete_object(Bucket=BUCKET_NAME, Key="raw/2024-03/customers.csv")
print("\nDeleted: raw/2024-03/customers.csv")

# Verify deletion
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="raw/")
print(f"Remaining raw objects: {response['KeyCount']}")

---
## Section 5: S3 Best Practices — CSV vs Parquet

One of the most impactful decisions for ML data pipelines is **file format**.

| Feature | CSV | Parquet |
|---------|-----|--------|
| Human readable | ✅ Yes | ❌ No (binary) |
| Compressed size | Large | **2-10x smaller** |
| Read speed | Slow (parse all text) | **Fast** (columnar) |
| Column selection | Must read entire row | **Read only needed columns** |
| Schema enforcement | ❌ None | ✅ Embedded schema |
| ML pipeline use | Prototyping | **Production** |

### Why Parquet for ML?
- **Smaller files** → less S3 storage cost, faster transfers
- **Columnar storage** → `pd.read_parquet(columns=['feature_1', 'feature_2'])` only reads what you need
- **Type preservation** → no more `"True"` string vs `True` boolean issues

Let's measure the difference:

In [ ]:
# Generate a realistic ML dataset for comparison
np.random.seed(42)
n_rows = 100_000

df_large = pd.DataFrame({
    'customer_id': range(1, n_rows + 1),
    'tenure_months': np.random.randint(1, 72, n_rows),
    'monthly_charges': np.round(np.random.uniform(18, 120, n_rows), 2),
    'total_charges': np.round(np.random.uniform(100, 8000, n_rows), 2),
    'contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_rows),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n_rows),
    'online_security': np.random.binomial(1, 0.35, n_rows),
    'tech_support': np.random.binomial(1, 0.30, n_rows),
    'streaming_tv': np.random.binomial(1, 0.45, n_rows),
    'churned': np.random.binomial(1, 0.25, n_rows),
})

print(f"Dataset: {df_large.shape[0]:,} rows x {df_large.shape[1]} columns")
print(f"Memory usage: {df_large.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

# Save as CSV
csv_path = Path(tempfile.mkdtemp()) / "data.csv"
start = time.time()
df_large.to_csv(csv_path, index=False)
csv_write_time = time.time() - start
csv_size = csv_path.stat().st_size

# Save as Parquet
parquet_path = csv_path.parent / "data.parquet"
start = time.time()
df_large.to_parquet(parquet_path, index=False)
parquet_write_time = time.time() - start
parquet_size = parquet_path.stat().st_size

# Compare
print(f"\n{'Format':<10} {'Size (MB)':<12} {'Write Time (s)':<16} {'Compression'}")
print(f"{'—'*50}")
print(f"{'CSV':<10} {csv_size/1024/1024:<12.2f} {csv_write_time:<16.3f} {'baseline'}")
print(f"{'Parquet':<10} {parquet_size/1024/1024:<12.2f} {parquet_write_time:<16.3f} {csv_size/parquet_size:.1f}x smaller")

In [ ]:
# Read speed comparison

# Full read
start = time.time()
_ = pd.read_csv(csv_path)
csv_read_full = time.time() - start

start = time.time()
_ = pd.read_parquet(parquet_path)
parquet_read_full = time.time() - start

# Selective column read (Parquet advantage)
cols = ['tenure_months', 'monthly_charges', 'churned']

start = time.time()
_ = pd.read_csv(csv_path, usecols=cols)
csv_read_cols = time.time() - start

start = time.time()
_ = pd.read_parquet(parquet_path, columns=cols)
parquet_read_cols = time.time() - start

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Size comparison
formats = ['CSV', 'Parquet']
sizes = [csv_size / 1024 / 1024, parquet_size / 1024 / 1024]
colors = ['#e74c3c', '#2ecc71']
axes[0].barh(formats, sizes, color=colors)
axes[0].set_xlabel('File Size (MB)')
axes[0].set_title('Storage Size: CSV vs Parquet')
for i, (fmt, size) in enumerate(zip(formats, sizes)):
    axes[0].text(size + 0.1, i, f'{size:.1f} MB', va='center')

# Read speed comparison
x = np.arange(2)
width = 0.35
csv_times = [csv_read_full, csv_read_cols]
parquet_times = [parquet_read_full, parquet_read_cols]
axes[1].bar(x - width/2, csv_times, width, label='CSV', color='#e74c3c')
axes[1].bar(x + width/2, parquet_times, width, label='Parquet', color='#2ecc71')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Full Read', f'3 Columns Only'])
axes[1].set_ylabel('Read Time (seconds)')
axes[1].set_title('Read Speed: CSV vs Parquet')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nParquet is {csv_read_full/parquet_read_full:.1f}x faster for full reads")
print(f"Parquet is {csv_read_cols/parquet_read_cols:.1f}x faster for selective column reads")

In [ ]:
# S3 cost estimation

def estimate_s3_cost(size_gb: float, reads_per_month: int = 1000, writes_per_month: int = 100) -> dict:
    """Estimate monthly S3 costs.
    
    Pricing (us-east-1, Standard tier):
    - Storage: $0.023/GB/month
    - PUT/COPY/POST: $0.005 per 1,000 requests
    - GET/SELECT: $0.0004 per 1,000 requests
    - Data transfer out: first 100 GB free, then $0.09/GB
    """
    storage_cost = size_gb * 0.023
    write_cost = (writes_per_month / 1000) * 0.005
    read_cost = (reads_per_month / 1000) * 0.0004
    
    return {
        'storage_cost': round(storage_cost, 4),
        'write_cost': round(write_cost, 4),
        'read_cost': round(read_cost, 4),
        'total_monthly': round(storage_cost + write_cost + read_cost, 4)
    }

# Compare costs for CSV vs Parquet
csv_gb = csv_size / (1024 ** 3)
parquet_gb = parquet_size / (1024 ** 3)

# Scale to a realistic dataset (1000x bigger)
scale = 1000
csv_cost = estimate_s3_cost(csv_gb * scale)
parquet_cost = estimate_s3_cost(parquet_gb * scale)

print(f"Monthly S3 cost estimate for {n_rows * scale / 1e6:.0f}M rows:")
print(f"  CSV:     ${csv_cost['total_monthly']:.2f}/month ({csv_gb * scale:.1f} GB)")
print(f"  Parquet: ${parquet_cost['total_monthly']:.2f}/month ({parquet_gb * scale:.1f} GB)")
print(f"  Savings: ${csv_cost['total_monthly'] - parquet_cost['total_monthly']:.2f}/month")

---
## Section 6: Boto3 Patterns — Reusable S3 Helper

In production, you'll use the same S3 operations repeatedly. Let's build a **reusable helper class**
that wraps common patterns with error handling.

In [ ]:
class S3Helper:
    """Reusable S3 helper for ML workflows.
    
    Wraps common operations with error handling, DataFrame support,
    and consistent logging.
    """
    
    def __init__(self, s3_client, bucket: str):
        self.s3 = s3_client
        self.bucket = bucket
    
    def upload_df(self, df: pd.DataFrame, key: str, fmt: str = "parquet") -> dict:
        """Upload a DataFrame to S3 in the specified format."""
        if fmt == "parquet":
            buffer = BytesIO()
            df.to_parquet(buffer, index=False)
            body = buffer.getvalue()
        elif fmt == "csv":
            body = df.to_csv(index=False).encode('utf-8')
        else:
            raise ValueError(f"Unsupported format: {fmt}")
        
        self.s3.put_object(Bucket=self.bucket, Key=key, Body=body)
        size_kb = len(body) / 1024
        print(f"Uploaded {key} ({size_kb:.1f} KB, {fmt}, {len(df):,} rows)")
        return {"key": key, "size_bytes": len(body), "rows": len(df)}
    
    def download_df(self, key: str, fmt: str = None) -> pd.DataFrame:
        """Download a DataFrame from S3."""
        if fmt is None:
            fmt = "parquet" if key.endswith(".parquet") else "csv"
        
        response = self.s3.get_object(Bucket=self.bucket, Key=key)
        body = response['Body'].read()
        
        if fmt == "parquet":
            df = pd.read_parquet(BytesIO(body))
        else:
            df = pd.read_csv(BytesIO(body))
        
        print(f"Downloaded {key} ({len(df):,} rows x {len(df.columns)} cols)")
        return df
    
    def list_keys(self, prefix: str = "") -> list[str]:
        """List all object keys under a prefix."""
        response = self.s3.list_objects_v2(Bucket=self.bucket, Prefix=prefix)
        return [obj['Key'] for obj in response.get('Contents', [])]
    
    def key_exists(self, key: str) -> bool:
        """Check if an object exists."""
        try:
            self.s3.head_object(Bucket=self.bucket, Key=key)
            return True
        except (FileNotFoundError, Exception):
            return False
    
    def copy_object(self, source_key: str, dest_key: str) -> None:
        """Copy an object within the same bucket."""
        response = self.s3.get_object(Bucket=self.bucket, Key=source_key)
        body = response['Body'].read()
        self.s3.put_object(Bucket=self.bucket, Key=dest_key, Body=body)
        print(f"Copied {source_key} -> {dest_key}")


# Demo the helper
helper = S3Helper(s3, BUCKET_NAME)

# Upload as Parquet (default)
helper.upload_df(df_large, "processed/customers_clean.parquet")

# Download
df_back = helper.download_df("processed/customers_clean.parquet")
print(f"\nRoundtrip check: shapes match = {df_large.shape == df_back.shape}")

# List all keys
print(f"\nAll keys in bucket:")
for key in helper.list_keys():
    print(f"  {key}")

# Check existence
print(f"\nprocessed/customers_clean.parquet exists: {helper.key_exists('processed/customers_clean.parquet')}")
print(f"models/v1/model.pkl exists: {helper.key_exists('models/v1/model.pkl')}")

---
## Section 7: AWS Lambda for ML Workflows

**AWS Lambda** runs code without provisioning servers. You write a function, AWS runs it when triggered.

### Lambda Anatomy

```python
def handler(event, context):
    """Every Lambda function has this signature.
    
    Args:
        event: Dict with trigger data (S3 event, API Gateway request, etc.)
        context: Runtime info (function name, memory, time remaining)
    
    Returns:
        Dict with statusCode and body (for API Gateway)
        or any serializable object (for other triggers)
    """
    return {"statusCode": 200, "body": "Hello from Lambda!"}
```

### ML Use Cases for Lambda

| Use Case | Trigger | What It Does |
|----------|---------|-------------|
| Data ingestion | S3 PutObject | Validate and preprocess new uploads |
| Lightweight inference | API Gateway | Score single predictions (< 15 min) |
| Alerting | CloudWatch alarm | Send Slack/email when model degrades |
| ETL orchestration | EventBridge schedule | Kick off nightly data refresh |

### Lambda Limitations for ML
- **Max runtime:** 15 minutes
- **Max memory:** 10 GB (no GPU)
- **Max package size:** 250 MB unzipped (50 MB zipped)
- **Cold starts:** First invocation takes 1-5 seconds

**Rule of thumb:** Lambda is great for data triggers and lightweight preprocessing.
Use SageMaker or ECS for heavy training or inference.

In [ ]:
# --- Lambda Handler: Data Validation on S3 Upload ---

def data_validation_handler(event, context=None):
    """Lambda triggered when a CSV is uploaded to raw/ prefix.
    
    Validates the data and writes results to processed/ if valid.
    This function works identically locally and in AWS Lambda.
    """
    # Parse S3 event
    record = event['Records'][0]
    bucket = record['s3']['bucket']['name']
    key = record['s3']['object']['key']
    
    print(f"Processing: s3://{bucket}/{key}")
    
    # Download the file
    response = s3.get_object(Bucket=bucket, Key=key)
    df = pd.read_csv(BytesIO(response['Body'].read()))
    print(f"  Loaded {len(df):,} rows x {len(df.columns)} columns")
    
    # --- Validation checks ---
    issues = []
    
    # Check 1: Required columns
    required_cols = {'customer_id', 'tenure_months', 'monthly_charges', 'churned'}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        issues.append(f"Missing columns: {missing_cols}")
    
    # Check 2: Null rates
    null_rates = df.isnull().mean()
    high_null_cols = null_rates[null_rates > 0.05].index.tolist()
    if high_null_cols:
        issues.append(f"High null rate (>5%): {high_null_cols}")
    
    # Check 3: Duplicate rows
    dup_rate = df.duplicated().mean()
    if dup_rate > 0.01:
        issues.append(f"Duplicate rate: {dup_rate:.1%}")
    
    # Check 4: Value ranges
    if 'tenure_months' in df.columns:
        if df['tenure_months'].min() < 0 or df['tenure_months'].max() > 120:
            issues.append("tenure_months out of expected range [0, 120]")
    
    if 'monthly_charges' in df.columns:
        if df['monthly_charges'].min() < 0 or df['monthly_charges'].max() > 500:
            issues.append("monthly_charges out of expected range [0, 500]")
    
    # Build result
    result = {
        "source_key": key,
        "rows": len(df),
        "columns": len(df.columns),
        "null_rate": float(df.isnull().mean().mean()),
        "duplicate_rate": float(dup_rate),
        "issues": issues,
        "status": "FAILED" if issues else "PASSED",
        "timestamp": datetime.now().isoformat()
    }
    
    # Save validation report
    report_key = key.replace("raw/", "logs/validation/").replace(".csv", "_report.json")
    s3.put_object(
        Bucket=bucket,
        Key=report_key,
        Body=json.dumps(result, indent=2).encode('utf-8')
    )
    print(f"  Validation: {result['status']} ({len(issues)} issues)")
    
    # If valid, save processed version as Parquet
    if result['status'] == 'PASSED':
        processed_key = key.replace("raw/", "processed/").replace(".csv", ".parquet")
        buffer = BytesIO()
        df.to_parquet(buffer, index=False)
        s3.put_object(Bucket=bucket, Key=processed_key, Body=buffer.getvalue())
        print(f"  Saved processed: {processed_key}")
    
    return result

print("Lambda handler defined. Let's test it locally!")

In [ ]:
# --- Test the Lambda handler locally ---

# Simulate an S3 PutObject event (this is what AWS sends to your Lambda)
s3_event = {
    "Records": [
        {
            "eventSource": "aws:s3",
            "eventName": "ObjectCreated:Put",
            "s3": {
                "bucket": {"name": BUCKET_NAME},
                "object": {"key": "raw/2024-01/customers.csv"}
            }
        }
    ]
}

# Run the handler
result = data_validation_handler(s3_event)
print(f"\nValidation Result:")
print(json.dumps(result, indent=2))

In [ ]:
# --- Test with bad data to see validation catch issues ---

# Create a dataset with intentional quality issues
n = 200
df_bad = pd.DataFrame({
    'customer_id': range(1, n + 1),
    'tenure_months': np.random.randint(-10, 200, n),   # out of range
    'monthly_charges': np.round(np.random.uniform(-5, 600, n), 2),  # out of range
    'churned': np.random.binomial(1, 0.25, n)
})

# Add 15% nulls
null_mask = np.random.random(n) < 0.15
df_bad.loc[null_mask, 'monthly_charges'] = np.nan

# Add 5% duplicates
dupes = df_bad.sample(int(n * 0.05), random_state=42)
df_bad = pd.concat([df_bad, dupes], ignore_index=True)

# Upload bad data
csv_bytes = df_bad.to_csv(index=False).encode('utf-8')
s3.put_object(Bucket=BUCKET_NAME, Key="raw/2024-04/bad_data.csv", Body=csv_bytes)

# Test validation
bad_event = {
    "Records": [{
        "eventSource": "aws:s3",
        "eventName": "ObjectCreated:Put",
        "s3": {
            "bucket": {"name": BUCKET_NAME},
            "object": {"key": "raw/2024-04/bad_data.csv"}
        }
    }]
}

result = data_validation_handler(bad_event)
print(f"\nValidation Result:")
print(json.dumps(result, indent=2))
print(f"\n^ The handler correctly caught the data quality issues!")

---
## Section 8: End-to-End Pipeline — S3 Event Trigger

Now let's put it all together: a complete data pipeline that simulates the AWS pattern:

```
Upload Raw CSV  →  Lambda Trigger  →  Validate  →  Preprocess  →  Feature Engineering  →  Save to features/
```

In production, S3 event notifications would trigger the Lambda automatically.
Here we simulate the trigger locally.

In [ ]:
def preprocess_handler(event, context=None):
    """Second stage: preprocess validated data."""
    record = event['Records'][0]
    bucket = record['s3']['bucket']['name']
    key = record['s3']['object']['key']
    
    print(f"\nPreprocessing: s3://{bucket}/{key}")
    
    # Download
    response = s3.get_object(Bucket=bucket, Key=key)
    df = pd.read_parquet(BytesIO(response['Body'].read()))
    original_rows = len(df)
    
    # Clean: remove duplicates
    df = df.drop_duplicates()
    
    # Clean: handle nulls (median imputation for numeric)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().any():
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"  Imputed {col} nulls with median={median_val:.2f}")
    
    print(f"  Rows: {original_rows:,} -> {len(df):,} (removed {original_rows - len(df)} dupes)")
    
    # Save preprocessed
    output_key = key.replace("processed/", "features/")
    buffer = BytesIO()
    df.to_parquet(buffer, index=False)
    s3.put_object(Bucket=bucket, Key=output_key, Body=buffer.getvalue())
    print(f"  Saved: {output_key}")
    
    return {"output_key": output_key, "rows": len(df)}


def run_pipeline(bucket: str, raw_key: str):
    """Orchestrate the full pipeline: validate → preprocess → feature store."""
    print("=" * 60)
    print(f"PIPELINE START: s3://{bucket}/{raw_key}")
    print("=" * 60)
    
    pipeline_start = time.time()
    
    # Stage 1: Validation
    print("\n--- Stage 1: Validation ---")
    s3_event = {
        "Records": [{
            "eventSource": "aws:s3",
            "eventName": "ObjectCreated:Put",
            "s3": {
                "bucket": {"name": bucket},
                "object": {"key": raw_key}
            }
        }]
    }
    val_result = data_validation_handler(s3_event)
    
    if val_result['status'] == 'FAILED':
        print(f"\nPIPELINE ABORTED: Validation failed")
        for issue in val_result['issues']:
            print(f"  - {issue}")
        return {"status": "FAILED", "reason": val_result['issues']}
    
    # Stage 2: Preprocessing
    print("\n--- Stage 2: Preprocessing ---")
    processed_key = raw_key.replace("raw/", "processed/").replace(".csv", ".parquet")
    preprocess_event = {
        "Records": [{
            "eventSource": "aws:s3",
            "eventName": "ObjectCreated:Put",
            "s3": {
                "bucket": {"name": bucket},
                "object": {"key": processed_key}
            }
        }]
    }
    prep_result = preprocess_handler(preprocess_event)
    
    # Pipeline summary
    elapsed = time.time() - pipeline_start
    print(f"\n{'=' * 60}")
    print(f"PIPELINE COMPLETE in {elapsed:.2f}s")
    print(f"  Input:     s3://{bucket}/{raw_key}")
    print(f"  Output:    s3://{bucket}/{prep_result['output_key']}")
    print(f"  Rows:      {prep_result['rows']:,}")
    print(f"{'=' * 60}")
    
    return {"status": "SUCCESS", "elapsed": elapsed, **prep_result}


# Run the pipeline
result = run_pipeline(BUCKET_NAME, "raw/2024-01/customers.csv")

In [ ]:
# Verify the output — read back the feature data
print("\n--- Final bucket contents ---")
response = s3.list_objects_v2(Bucket=BUCKET_NAME)
for obj in response.get('Contents', []):
    print(f"  {obj['Key']:55s}  {obj['Size']:>10,} bytes")

# Read the feature file
feature_keys = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].startswith('features/')]
if feature_keys:
    resp = s3.get_object(Bucket=BUCKET_NAME, Key=feature_keys[0])
    df_features = pd.read_parquet(BytesIO(resp['Body'].read()))
    print(f"\nFeature data: {df_features.shape}")
    print(f"Null values: {df_features.isnull().sum().sum()}")
    print(f"Duplicates: {df_features.duplicated().sum()}")
    print(df_features.head())

---
## Section 9: Exercises

### Exercise 1: S3 File Manager
Build a function `s3_sync(local_dir, bucket, prefix)` that:
1. Scans a local directory for files
2. Compares against S3 (by key name)
3. Uploads only new or modified files
4. Returns a summary: uploaded, skipped, errors

### Exercise 2: Pipeline Metrics Logger
Extend the pipeline to log metrics (timing, row counts, error rates) to a
`logs/pipeline_metrics.json` file in S3. Each pipeline run should append an entry.

### Exercise 3: S3 Cost Estimator
Write a function that scans all objects in a bucket and produces a cost report:
- Total storage by prefix (raw/, processed/, models/)
- Estimated monthly cost per prefix
- Recommendations (e.g., "move 2024-01 raw data to Glacier")

In [ ]:
# Exercise 1: S3 File Manager

# TODO: Implement s3_sync(local_dir, bucket, prefix)

# def s3_sync(local_dir: str, bucket: str, prefix: str = "") -> dict:
#     """Sync a local directory to S3 (upload new/modified files)."""
#     local_path = Path(local_dir)
#     uploaded, skipped, errors = 0, 0, 0
#
#     # Get existing S3 keys
#     existing = set()
#     response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
#     for obj in response.get('Contents', []):
#         existing.add(obj['Key'])
#
#     # Scan local files
#     for filepath in local_path.rglob('*'):
#         if filepath.is_file():
#             relative = str(filepath.relative_to(local_path))
#             s3_key = f"{prefix}{relative}" if prefix else relative
#
#             if s3_key in existing:
#                 skipped += 1
#             else:
#                 try:
#                     s3.upload_file(Filename=str(filepath), Bucket=bucket, Key=s3_key)
#                     uploaded += 1
#                 except Exception as e:
#                     print(f"Error uploading {s3_key}: {e}")
#                     errors += 1
#
#     return {"uploaded": uploaded, "skipped": skipped, "errors": errors}
#
# # Test it
# test_dir = Path(tempfile.mkdtemp())
# (test_dir / "file1.txt").write_text("hello")
# (test_dir / "file2.txt").write_text("world")
# result = s3_sync(str(test_dir), BUCKET_NAME, prefix="sync-test/")
# print(result)

In [ ]:
# Exercise 2: Pipeline Metrics Logger

# TODO: Create a PipelineMetricsLogger class that:
# - Logs start/end time, row counts, stage durations
# - Stores metrics in S3 as JSON
# - Can retrieve and plot historical metrics

# class PipelineMetricsLogger:
#     def __init__(self, s3_client, bucket, metrics_key="logs/pipeline_metrics.json"):
#         self.s3 = s3_client
#         self.bucket = bucket
#         self.metrics_key = metrics_key
#         self.runs = self._load_existing()
#
#     def _load_existing(self):
#         try:
#             resp = self.s3.get_object(Bucket=self.bucket, Key=self.metrics_key)
#             return json.loads(resp['Body'].read())
#         except (FileNotFoundError, Exception):
#             return []
#
#     def log_run(self, run_data: dict):
#         run_data['logged_at'] = datetime.now().isoformat()
#         self.runs.append(run_data)
#         body = json.dumps(self.runs, indent=2).encode('utf-8')
#         self.s3.put_object(Bucket=self.bucket, Key=self.metrics_key, Body=body)
#
#     def get_history(self) -> pd.DataFrame:
#         return pd.DataFrame(self.runs)

In [ ]:
# Exercise 3: S3 Cost Estimator

# TODO: Write estimate_bucket_cost(s3_client, bucket) that produces
# a cost breakdown by prefix

# def estimate_bucket_cost(s3_client, bucket: str) -> pd.DataFrame:
#     """Estimate S3 storage costs by prefix."""
#     response = s3_client.list_objects_v2(Bucket=bucket)
#     objects = response.get('Contents', [])
#
#     prefix_sizes = {}
#     for obj in objects:
#         prefix = obj['Key'].split('/')[0] + '/'
#         prefix_sizes[prefix] = prefix_sizes.get(prefix, 0) + obj['Size']
#
#     rows = []
#     for prefix, size_bytes in prefix_sizes.items():
#         size_gb = size_bytes / (1024 ** 3)
#         monthly_cost = size_gb * 0.023
#         rows.append({
#             'prefix': prefix,
#             'size_mb': round(size_bytes / 1024 / 1024, 2),
#             'monthly_cost': round(monthly_cost, 4),
#         })
#
#     df = pd.DataFrame(rows).sort_values('size_mb', ascending=False)
#     print(f"Total monthly estimate: ${df['monthly_cost'].sum():.4f}")
#     return df
#
# estimate_bucket_cost(s3, BUCKET_NAME)

---
## Section 10: Key Takeaways

1. **AWS = APIs:** Everything in AWS is an HTTP API. Boto3 is your Python interface to all of them.

2. **IAM first:** Always start with least-privilege policies. Scope S3 access to specific buckets and prefixes.

3. **S3 organization matters:** Use prefixes (`raw/`, `processed/`, `features/`, `models/`) to organize ML data. This maps directly to your pipeline stages.

4. **Parquet > CSV for production:** 2-10x smaller files, faster reads, embedded schema. Always convert to Parquet after initial ingestion.

5. **Lambda for glue logic:** Perfect for data validation triggers, lightweight preprocessing, and pipeline orchestration. Not suitable for heavy ML training.

6. **Test locally first:** Use `LocalS3Simulator` to develop and test your pipelines before deploying to AWS. The API is the same.

### What's Next
In **Notebook 02**, we'll use SageMaker to train models at scale — building on the S3 data pipeline patterns from this notebook.

In [ ]:
# Cleanup — remove temporary local S3 data
if not USE_REAL_AWS:
    s3.cleanup()